# Baseline Classifier (MFCC Summary Features)

Trains a simple baseline model on MFCC summary features using a session-based split
(Sessions 1-4 = train, Session 5 = test). Gender is excluded.

In [19]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score

repo_root = Path.cwd().parents[1]
csv_path = repo_root / "extracted_features" / "mfcc" / "mfcc_features.csv"
csv_path

WindowsPath('f:/Speech-Emotion-Recognition/extracted_features/mfcc/mfcc_features.csv')

In [20]:
df = pd.read_csv(csv_path)

# Safety filter: ensure valid labels
df = df[(df["emotion"] != "xxx") & (df["agreement"] > 1)].copy()
df.shape

(7532, 1157)

In [ ]:
# Define metadata columns 
metadata_cols = [
    "path",
    "session",
    "method",
    "gender",
    "emotion",
    "n_annotators",
    "agreement",
]

feature_cols = [c for c in df.columns if c not in metadata_cols]
X = df[feature_cols].copy()
y = df["emotion"].copy()

# Drop rows with any missing values in features
mask = X.notna().all(axis=1)
X = X.loc[mask]
y = y.loc[mask]
df = df.loc[mask]

X.shape, y.shape

((7532, 1150), (7532,))

In [22]:
# Session-based split: Sessions 1-4 train, Session 5 test
train_mask = df["session"].isin([1, 2, 3, 4])
test_mask = df["session"].isin([5])

X_train = X[train_mask]
y_train = y[train_mask]
X_test = X[test_mask]
y_test = y[test_mask]

X_train.shape, X_test.shape

((5882, 1150), (1650, 1150))

In [23]:
# Baseline model (no normalization)
clf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced",
)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="macro")

print(f"Accuracy: {acc:.4f}")
print(f"Macro F1:  {f1:.4f}")
print("\nClassification report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.4024
Macro F1:  0.2619

Classification report:

              precision    recall  f1-score   support

         ang       0.55      0.45      0.50       170
         exc       0.62      0.10      0.17       299
         fea       0.00      0.00      0.00        10
         fru       0.31      0.63      0.42       381
         hap       0.00      0.00      0.00       143
         neu       0.38      0.45      0.41       384
         sad       0.62      0.58      0.60       245
         sur       0.00      0.00      0.00        18

    accuracy                           0.40      1650
   macro avg       0.31      0.28      0.26      1650
weighted avg       0.42      0.40      0.36      1650



f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape